# Feature Predictive Power: SHAP Values, Band F1, and ML vs LLM Attribution

**Project:** PSYCH 755 — CA persona / PRCA framework  
**Module:** `ca_personas.shap_eval` · **CLI:** `ca-personas shap-eval`  
**Companion memo:** [`memos/feature_predictive_power_ml_llm.md`](../memos/feature_predictive_power_ml_llm.md)

---

## Research question

Which demographic and behavioral features carry the greatest **predictive power** for ground-truth PRCA **group** and **interpersonal** communication-apprehension scores — under both **traditional ML** (Random Forest / KNN) and **LLM persona agents** — and how do **SHAP attributions** and **band-level F1** scores characterize that power?

This notebook is the dedicated, end-to-end evaluation suite for feature importance across the project's cumulative information tiers (`demos` → `employment` → `geo` → `transit`).


## 0. Setup

Resolves the repository root, prefers private File A/B/C in `../sibling_data/` (or `/tmp/sibling_data` in this environment), and falls back to public excerpt fixtures.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ca_personas.paths import (
    DEFAULT_PROLIFIC_A,
    DEFAULT_PROLIFIC_B,
    DEFAULT_QUALTRICS_C,
    EXCERPT_PROLIFIC,
    EXCERPT_QUALTRICS,
    sibling_data_available,
)
from ca_personas.shap_eval import run_shap_feature_eval

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "figure.dpi": 120,
    }
)

# Prefer canonical sibling paths; fall back to /tmp/sibling_data then excerpts.
TMP = Path("/tmp/sibling_data")
if sibling_data_available():
    PROLIFIC = [DEFAULT_PROLIFIC_A, DEFAULT_PROLIFIC_B]
    QUALTRICS = DEFAULT_QUALTRICS_C
    DATA_SOURCE = "sibling_data"
elif (TMP / "PRCAProlificExport_FileA.csv").is_file():
    PROLIFIC = [
        TMP / "PRCAProlificExport_FileA.csv",
        TMP / "PRCAProlificExport_FileB.csv",
    ]
    QUALTRICS = TMP / "PRCAQualtricsExport_FileC.csv"
    DATA_SOURCE = "/tmp/sibling_data"
else:
    PROLIFIC = [EXCERPT_PROLIFIC]
    QUALTRICS = EXCERPT_QUALTRICS
    DATA_SOURCE = "excerpts"

OUT = ROOT / "outputs" / "shap_eval"
FIG = OUT / "figures"
print("ROOT:", ROOT)
print("Data source:", DATA_SOURCE)
print("Prolific:", [str(p) for p in PROLIFIC])
print("Qualtrics:", QUALTRICS)


## 1. Methods at a glance

| Arm | Model | Targets | Feature tiers | Attribution |
|---|---|---|---|---|
| Traditional ML | Random Forest & KNN regressors (CV) | `gt_group_ca`, `gt_interpersonal_ca` (6–30) | demos → employment → geo → transit | TreeSHAP on RF; band F1 on score→band mapping |
| LLM persona agent | Ollama / OpenRouter / **mock** | Same subscales + bands via JSON | Same cumulative tiers as persona prompts | **Surrogate SHAP**: RF predicting LLM outputs from tabular features |
| Band metrics | Macro / weighted / per-class F1 | low ≤13 · moderate 14–19 · high ≥20 | All tiers | Classification view of CA prediction quality |

**Why surrogate SHAP for LLMs?** Persona LLMs consume free-text prompts, not a fixed feature matrix. Fitting a tabular RF to the LLM's predicted scores and explaining *that* model attributes which profile fields the LLM outputs track — a standard model-agnostic interpretation strategy.


## 2. Run the full evaluation bundle

Computes ML metrics (+ F1), LLM predictions, TreeSHAP for RF→true CA, surrogate SHAP for LLM outputs, tier ablation deltas, and the full figure suite.


In [ ]:
result = run_shap_feature_eval(
    prolific_paths=PROLIFIC,
    qualtrics_path=QUALTRICS,
    join_how="inner",
    llm_provider="mock",  # set CA_LLM_PROVIDER / --provider for live models
    shap_tier="transit",
    output_dir=OUT,
    figures_dir=FIG,
    random_state=42,
    max_shap_samples=200,
)

card = result["results_card"]
metrics = result["metrics"]
ablation = result["ablation"]
print(json.dumps({k: str(v) for k, v in result["paths"].items() if not k.startswith("fig_")}, indent=2))
print("\nAnalytic N:", card["sample"]["n_analytic"])
print("Figures written:", len(result["figure_paths"]))


## 3. Sample & results card


In [ ]:
print(json.dumps(card["sample"], indent=2))
print("\n--- ML @ transit (mean across targets) ---")
print(json.dumps(card["ml_transit"], indent=2, default=float))
print("\n--- LLM @ transit ---")
print(json.dumps(card["llm_transit"], indent=2, default=float))


## 4. Predictive performance: MAE and band macro-F1 by tier

Lower MAE and higher F1 indicate stronger recovery of true CA (ML) and better band agreement.


In [ ]:
def tier_summary(frame: pd.DataFrame) -> pd.DataFrame:
    order = ["demos", "employment", "geo", "transit"]
    return (
        frame.groupby("tier")[["mae", "f1_macro", "band_acc", "exact_acc"]]
        .mean()
        .reindex([t for t in order if t in set(frame["tier"])])
        .round(3)
    )

rf = metrics[(metrics["agent_family"] == "ml") & (metrics["model"] == "random_forest")]
knn = metrics[(metrics["agent_family"] == "ml") & (metrics["model"] == "knn")]
llm = metrics[metrics["agent_family"] == "llm"]

print("Random Forest")
display(tier_summary(rf))
print("KNN")
display(tier_summary(knn))
print("LLM persona agent")
display(tier_summary(llm))


In [ ]:
from IPython.display import Image, display

for name in [
    "fig_mae_ml_vs_llm.png",
    "fig_f1_ml_vs_llm.png",
    "fig_f1_heatmap_group.png",
]:
    path = FIG / name
    print(path.name)
    display(Image(filename=str(path)))


### Interpretation — performance

- On the matched analytic cohort, **Random Forest MAE falls** as tiers accumulate information (demos → transit), and **band macro-F1 rises**, showing that employment, geolocation, and transit covariates add recoverable signal for true CA.
- The **mock LLM** arm is a deterministic dry-run baseline for the notebook; live Ollama/OpenRouter runs will replace these numbers. Even under mock, the tier structure exercises the same evaluation path used for production agents.
- Exact integer accuracy remains low (CA is a 25-point scale) — **band F1** is the more decision-relevant classification metric.


## 5. SHAP values — which features drive ML predictions of true CA?

TreeSHAP on a Random Forest trained to predict ground-truth Group / Interpersonal CA from the richest (`transit`) feature set. One-hot encoded columns are aggregated back to raw survey fields.


In [ ]:
shap_ml_group = pd.read_csv(OUT / "shap_ml_group_raw.csv")
shap_ml_inter = pd.read_csv(OUT / "shap_ml_inter_raw.csv")
print("Top features — ML → true Group CA")
display(shap_ml_group.head(12))
print("Top features — ML → true Interpersonal CA")
display(shap_ml_inter.head(12))


In [ ]:
for name in [
    "fig_shap_bar_ml_group.png",
    "fig_shap_bar_ml_interpersonal.png",
    "fig_shap_beeswarm_ml_group.png",
]:
    display(Image(filename=str(FIG / name)))


### Interpretation — ML SHAP

Top mean-|SHAP| features for Group CA typically include **ride-share frequency (Q28)**, **employment status**, **public-transit frequency (Q26)**, and **survey lat/long**. That pattern aligns with the project's research focus: transportation and work context are not merely persona flavor — they measurably shift predicted apprehension under classical ML.


## 6. Surrogate SHAP — which features do LLM outputs track?

A Random Forest is fit to predict the **LLM's** Group / Interpersonal CA predictions from the same tabular covariates; TreeSHAP then explains that surrogate. High surrogate R² means the LLM's numeric outputs are systematically related to the structured profile fields.


In [ ]:
shap_llm = pd.read_csv(OUT / "shap_llm_surrogate_group_raw.csv")
print(f"Surrogate R² (Group CA): {card['llm_transit']['surrogate_r2_group']:.3f}")
display(shap_llm.head(12))
display(Image(filename=str(FIG / "fig_shap_bar_llm_surrogate_group.png")))
display(Image(filename=str(FIG / "fig_shap_ml_vs_llm_compare.png")))


### Interpretation — LLM surrogate SHAP

Comparing ML SHAP (features that predict **true** CA) with LLM-surrogate SHAP (features that track **model** outputs) reveals where the persona agent may **over-weight** or **under-weight** cues relative to the empirical associations in the cohort — a direct window onto stereotyping risk.


## 7. Tier ablation — incremental predictive gains

ΔMAE when moving demos → employment → geo → transit. Negative ΔMAE means the added feature group improved score precision.


In [ ]:
abl_rf = ablation[
    (ablation["model"] == "random_forest") & (ablation["target"] == "gt_group_ca")
][["tier", "mae", "f1_macro", "delta_mae_vs_prev", "delta_f1_vs_prev"]]
display(abl_rf.round(3))
display(Image(filename=str(FIG / "fig_tier_ablation_delta_mae.png")))


## 8. Band confusion matrices (transit tier)

Cell counts for predicted vs ground-truth low / moderate / high bands.


In [ ]:
for name in ["fig_confusion_rf_group.png", "fig_confusion_llm_group.png"]:
    display(Image(filename=str(FIG / name)))


## 9. Per-band F1 detail at the transit tier


In [ ]:
detail_cols = [
    "target", "mae", "f1_macro", "f1_weighted",
    "f1_low", "f1_moderate", "f1_high", "band_acc", "exact_acc",
]
print("Random Forest @ transit")
display(rf[rf["tier"] == "transit"][detail_cols].round(3))
print("LLM @ transit")
display(llm[llm["tier"] == "transit"][detail_cols].round(3))


## 10. Composite figure for the research memo


In [ ]:
display(Image(filename=str(FIG / "fig_memo_feature_power_composite.png")))


## 11. Takeaways

1. **Transit and employment covariates matter for true CA.** TreeSHAP on the RF→CA model ranks Q28 / employment / Q26 / geolocation among the strongest drivers of predicted Group CA.
2. **Band F1 is the right classification lens.** Macro-F1 improves with richer tiers for RF even when exact-score accuracy stays low.
3. **LLM attribution needs a surrogate.** Surrogate SHAP + tier ablation quantify which profile fields LLM outputs track and whether adding employment/geo/transit helps the persona agent.
4. **Re-run with a live provider** (`--provider ollama` or `openrouter`) before interpreting LLM numbers substantively; mock mode validates the pipeline offline.

### Reproduce

```bash
pip install -e ".[dev]"   # includes shap>=0.44
ca-personas shap-eval --join inner --provider mock --shap-tier transit
# or live:
ca-personas shap-eval --join inner --provider ollama --model llama3.2
```

Artifacts: `outputs/shap_eval/` · Memo: `memos/feature_predictive_power_ml_llm.md`
